In [343]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [113]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, LSTM, Dense, Reshape,
    MaxPooling1D, Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [114]:

# ============================================================
# 하이퍼파라미터 상수 정의
# ============================================================
N_TRIALS   = 30        # 독립 실행 횟수 (N_F = 30)
BATCH_SIZE = 2 ** 15   # 배치 사이즈 = 32,768
MAX_EPOCHS = 100       # 최대 epoch 수
PATIENCE   = 5         # Early stopping patience
VAL_SPLIT  = 0.3       # Validation set 비율 (30%)


# 한국 시장 OP/TP 설정
PERIODS = [
    {
        "op_start": "1993-01-31",
        "op_end"  : "2002-12-31",
        "tp_start": "2003-01-31",
        "tp_end"  : "2012-12-31"
    },
    {
        "op_start": "1993-01-31",
        "op_end"  : "2012-12-31",
        "tp_start": "2013-01-31",
        "tp_end"  : "2023-12-31"
    }
]

# ============================================================
# STEP 0: 데이터 로드
# ============================================================
def load_data():
    """
    원시 데이터 로드 및 무위험 수익률 계산
    """
    monthly_return = pd.read_csv(
        r"data\monthly_return.csv",
        index_col="Date",
        parse_dates=True
    )
    market_cap = pd.read_csv(
        r"data\mkt_cap.csv",
        index_col="Date",
        parse_dates=True
    )
    cd91 = pd.read_csv(
        r"data\cd91.csv",
        index_col="Date",
        parse_dates=True
    )
    activate = pd.read_csv(
        r"data\activate.csv",
        index_col="Date",
        parse_dates=True
    )

    monthly_return = monthly_return / 100
    cd91           = cd91 / 100

    risk_free = (1 + cd91) ** (1 / 12) - 1

    print("=" * 55)
    print("데이터 로드 완료")
    print(f"  수익률  : {monthly_return.shape} "
          f"({monthly_return.index[0].date()} ~ "
          f"{monthly_return.index[-1].date()})")
    print(f"  시가총액: {market_cap.shape}")
    print(f"  무위험률: {risk_free.shape}")
    print(f"  거래상태: {activate.shape}")
    print("=" * 55)

    return monthly_return, market_cap, risk_free, activate


In [ ]:

# ============================================================
# STEP 1: CR1 ~ CR12 계산
# ============================================================
def calculate_cumulative_returns(monthly_return):
    """
    각 월(t)에 대해 CR1 ~ CR12 계산
    """
    n_dates = len(monthly_return)
    CR      = {}

    for k in range(1, 13):
        cr_k = pd.DataFrame(
            np.nan,
            index  =monthly_return.index,
            columns=monthly_return.columns,
            dtype  =float
        )

        for t_idx in range(n_dates):
            start_idx = t_idx - k
            end_idx   = t_idx 

            if start_idx < 0:
                continue

            window     = monthly_return.iloc[start_idx:end_idx]
            cumulative = (1 + window).prod(axis=0) - 1
            cr_k.iloc[t_idx] = cumulative.values

        CR[k] = cr_k
        print(f"  CR{k:>2} 계산 완료")

    print("CR1 ~ CR12 계산 완료\n")
    return CR


# ============================================================
# STEP 2: 정규화된 초과 수익률 계산 (목표 변수)
# ============================================================
def calculate_normalized_returns(monthly_return, risk_free):
    """
    정규화된 초과 수익률 계산
    r_Norm(i,t) = Φ⁻¹(rank(i,t) / (N_t + 1))
    """
    if isinstance(risk_free, pd.DataFrame):
        rf_series = risk_free.iloc[:, 0]
    else:
        rf_series = risk_free

    excess_return = monthly_return.sub(rf_series, axis=0)

    r_norm = pd.DataFrame(
        np.nan,
        index  =excess_return.index,
        columns=excess_return.columns,
        dtype  =float
    )

    for t_idx in range(len(excess_return)):
        row        = excess_return.iloc[t_idx]
        valid_mask = row.notna()
        valid_row  = row[valid_mask]

        if len(valid_row) == 0:
            continue

        N_t           = len(valid_row)
        ranks         = valid_row.rank(method='average') 
        r_norm_values = norm.ppf(ranks / (N_t + 1)) 

        r_norm.loc[excess_return.index[t_idx], valid_mask] = r_norm_values

    valid_ratio = r_norm.notna().values.mean()
    print(f"정규화 수익률 계산 완료 (유효값 비율: {valid_ratio:.2%})\n")

    return r_norm


# ============================================================
# STEP 3: CNNLSTM 모델 구성
# ============================================================
def build_cnnlstm_model():
    """
    Murray et al. (2024) 사양 및 옵티마이저 표준값 적용 CNNLSTM 모델 구성
    """
    model = Sequential([
        Reshape(target_shape=(12, 1), input_shape=(12,)),
        Conv1D(filters=64, kernel_size=5, strides=1, padding='valid', activation='relu'),
        MaxPooling1D(pool_size=2, strides=None, padding='valid'),
        LSTM(units=64, return_sequences=False),
        Dense(64, activation='relu'),
        Dropout(rate=0.2),
        Dense(1, activation='linear')
    ])

    optimizer = Adam(
        learning_rate=0.001,
        beta_1       =0.9,
        beta_2       =0.99, # 수정됨
        epsilon      =1e-7
    )

    model.compile(
        optimizer=optimizer,
        loss     ='mse'
    )

    return model


# ============================================================
# STEP 4: 데이터셋 구성
# ============================================================
def prepare_dataset(CR, r_norm, start_date, end_date,
                    market_cap=None, activate=None):
    """
    학습/예측용 데이터셋 구성
    """
    dates = r_norm.loc[start_date:end_date].index

    X_list     = []
    y_list     = []
    index_info = []

    for date in dates:
        for stock in r_norm.columns:
            y_val = r_norm.loc[date, stock]
            if pd.isna(y_val):
                continue

            if market_cap is not None:
                if pd.isna(market_cap.loc[date, stock]):
                    continue

            if activate is not None:
                if date in activate.index and stock in activate.columns:
                    status = activate.loc[date, stock]
                    if status in ('거래정지', '거래중단'):
                        continue

            cr_values = []
            valid     = True

            for k in range(1, 13):
                cr_val = CR[k].loc[date, stock]
                if pd.isna(cr_val):
                    valid = False
                    break
                cr_values.append(cr_val)

            if not valid:
                continue

            X_list.append(cr_values)
            y_list.append(y_val)
            index_info.append((date, stock))

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)

    return X, y, index_info


# ============================================================
# STEP 5: EWPM 가중치 계산 및 검증
# ============================================================
def calculate_sample_weights(index_info):
    """
    EWPM (Equal Weight Per Month) 가중치 계산
    """
    date_counts = {}
    for date, stock in index_info:
        date_counts[date] = date_counts.get(date, 0) + 1

    n_months = len(date_counts)

    weights = np.array(
        [1.0 / (n_months * date_counts[date]) for date, _ in index_info],
        dtype=np.float32
    )

    weights = weights / weights.sum() * len(weights)
    return weights


def verify_ewpm_weights(weights, index_info, label=""):
    """
    EWPM 가중치 균등성 검증
    """
    date_weight_sum = {}
    for w, (date, _) in zip(weights, index_info):
        date_weight_sum[date] = date_weight_sum.get(date, 0.0) + w

    sums = np.array(list(date_weight_sum.values()))
    cv   = sums.std() / sums.mean()

    status = "✅ 균등" if cv < 1e-4 else "❌ 불균등"
    print(f"  [{label}] 월별 가중치 합 → "
          f"mean={sums.mean():.4f}, "
          f"std={sums.std():.2e}, "
          f"CV={cv:.2e}  {status}")


# ============================================================
# STEP 6: Train / Validation 시계열 분리 (수정됨)
# ============================================================
def split_train_validation(X, y, index_info, val_split=VAL_SPLIT):
    """
    데이터 누수를 막기 위해 시점(Month)을 기준으로 Train / Validation 분할
    """
    # 1. 고유 날짜 추출 및 정렬 (시계열 순서 보장)
    unique_dates = sorted(list(dict.fromkeys([date for date, stock in index_info])))
    
    # 2. 날짜 기준으로 분할 인덱스 계산
    split_date_idx = int(len(unique_dates) * (1 - val_split))
    split_date = unique_dates[split_date_idx]
    
    # 3. 해당 날짜가 시작되는 첫 번째 샘플의 위치(row index) 찾기
    split_idx = next(i for i, (date, _) in enumerate(index_info) if date >= split_date)

    # 분리
    X_train      = X[:split_idx]
    y_train      = y[:split_idx]
    index_train  = index_info[:split_idx]

    X_val        = X[split_idx:]
    y_val        = y[split_idx:]
    index_val    = index_info[split_idx:]

    # 각 subset에 대해 독립적으로 EWPM 가중치 계산
    w_train = calculate_sample_weights(index_train)
    w_val   = calculate_sample_weights(index_val)

    print(f"  Train      : {len(X_train):>8,} 샘플")
    print(f"  Validation : {len(X_val):>8,} 샘플")

    verify_ewpm_weights(w_train, index_train, label="Train")
    verify_ewpm_weights(w_val,   index_val,   label="Validation")

    return (X_train, y_train, w_train,
            X_val,   y_val,   w_val,
            index_train, index_val)


# ============================================================
# STEP 7: MLER 계산 메인 함수
# ============================================================
def calculate_mler(CR, r_norm, market_cap, activate=None,
                   periods=PERIODS, n_trials=N_TRIALS):
    """
    MLER (ML-based Expected Return) 계산
    """
    all_predictions = {}

    device = (
        "/GPU:0"
        if tf.config.list_physical_devices('GPU')
        else "/CPU:0"
    )
    print(f"학습 디바이스: {device}\n")

    for period_idx, period in enumerate(periods):
        op_start = period["op_start"]
        op_end   = period["op_end"]
        tp_start = period["tp_start"]
        tp_end   = period["tp_end"]

        print(f"\n{'='*55}")
        print(f"Period {period_idx + 1}/{len(periods)}")
        print(f"  OP : {op_start} ~ {op_end}")
        print(f"  TP : {tp_start} ~ {tp_end}")
        print(f"{'='*55}")

        print("\n[1] 학습 데이터 준비 중...")
        X_all, y_all, index_all = prepare_dataset(
            CR, r_norm, op_start, op_end,
            market_cap=market_cap, activate=activate
        )
        print(f"  전체 샘플 수: {len(X_all):,}")

        print("\n[2] Train / Validation 분리 및 EWPM 가중치 계산...")
        (X_train, y_train, w_train,
         X_val,   y_val,   w_val,
         index_train, index_val) = split_train_validation(
            X_all, y_all, index_all, val_split=VAL_SPLIT
        )

        print("\n[3] 예측 데이터 준비 중...")
        X_test, _, test_index = prepare_dataset(
            CR, r_norm, tp_start, tp_end,
            market_cap=market_cap, activate=activate
        )

        print(f"\n[4] {n_trials}회 독립 실행 시작...")
        trial_predictions = []

        for trial in range(n_trials):
            tf.keras.backend.clear_session()
            tf.random.set_seed(trial)
            np.random.seed(trial)

            with tf.device(device):
                model = build_cnnlstm_model()

                early_stopping = EarlyStopping(
                    monitor             ='val_loss',
                    patience            =PATIENCE,
                    restore_best_weights=True,
                    verbose             =0
                )

                model.fit(
                    X_train,
                    y_train,
                    sample_weight   =w_train,
                    epochs          =MAX_EPOCHS,
                    batch_size      =BATCH_SIZE,
                    validation_data =(
                        X_val,
                        y_val,
                        w_val
                    ),
                    callbacks=[early_stopping], 
                    verbose  =0
                )

                pred = model.predict(X_test, verbose=0).flatten()

            trial_predictions.append(pred)

            stopped_epoch = early_stopping.stopped_epoch
            if stopped_epoch > 0:
                final_epoch = stopped_epoch + 1
            else:
                final_epoch = len(model.history.history['loss'])
            print(f"  Trial {trial+1:02d}/{n_trials} 완료 "
                  f"(epoch {final_epoch:>3})")

        mean_predictions = np.mean(trial_predictions, axis=0)

        for idx, (date, stock) in enumerate(test_index):
            if date not in all_predictions:
                all_predictions[date] = {}
            all_predictions[date][stock] = mean_predictions[idx]

        print(f"\nTP 기간 MLER 계산 완료: {tp_start} ~ {tp_end}")

    mler_df = pd.DataFrame(all_predictions).T
    mler_df.index.name = "Date"
    mler_df            = mler_df.sort_index()

    return mler_df


# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"GPU 사용 가능: {[g.name for g in gpus]}")
        except RuntimeError as e:
            print(f"GPU 설정 오류: {e}")
    else:
        print("GPU 없음 → CPU 실행")

    monthly_return, market_cap, risk_free, activate = load_data()

    print("\nCR1 ~ CR12 계산 중...")
    CR = calculate_cumulative_returns(monthly_return)

    print("정규화 수익률 계산 중...")
    r_norm = calculate_normalized_returns(monthly_return, risk_free)

    print("MLER 계산 시작...\n")
    mler = calculate_mler(
        CR, r_norm, market_cap, activate=activate,
        periods=PERIODS, n_trials=N_TRIALS
    )

    print("\n" + "=" * 55)
    print(f"  Shape : {mler.shape}")
    print(f"  기간  : {mler.index[0].date()} ~ {mler.index[-1].date()}")
    print("=" * 55)
    print("\n[MLER 기술 통계]")
    print(mler.stack().describe())

    mler.to_csv(r"result\mler.csv")

In [150]:
mler = pd.read_csv("result\mler.csv",index_col="Date",parse_dates=True)
mler

,삼성전자,SK하이닉스,현대차,기아,두산에너빌리티,한화에어로스페이스,현대모비스,신한지주,한화오션,고려아연,...,동인기연,비아이매트릭스,제이엔비,캡스톤파트너스,파라택시스이더리움,쏘닉스,KB제27호스팩,유진테크놀로지,세니젠,한국제13호스팩
Date,,,,,,,,,,,,,,,,,,,,,
2003-01-31,0.007528,0.017508,0.011549,0.003093,0.006893,-0.013492,0.006814,0.016645,0.002353,-0.009265,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003-02-28,0.009260,0.020396,0.014272,0.008604,0.012026,-0.008357,0.004610,0.010000,-0.001621,-0.000240,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003-03-31,0.013703,NaN,0.012072,0.008200,0.011140,0.004440,0.004607,0.010054,0.001705,-0.009864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003-04-30,0.009869,0.035544,0.011049,0.009988,0.007406,0.014705,0.012947,0.018396,0.004502,-0.009602,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003-05-31,0.008805,0.021316,-0.000695,-0.003442,-0.003814,0.022138,0.000482,0.005433,0.002315,-0.010966,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-31,0.106896,0.082754,0.108124,0.113725,0.084464,0.069385,0.100086,0.010604,-0.048348,0.003614,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-09-30,0.092143,0.094837,0.094709,0.098114,0.076761,0.080701,0.106294,0.010654,0.049501,0.023035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-10-31,0.083250,0.110228,0.098699,0.098064,0.094970,0.098938,0.118625,0.028735,0.071221,0.042161,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [151]:

# ============================================================
# Table 1: ML-Based Forecast Summary Statistics
# ============================================================
# 논문 방법론:
#   1) 매월 t: MLER의 횡단면(cross-sectional) 통계 계산
#   2) 해당 통계들의 시계열(time-series) 평균 보고

def compute_table1(mler, periods_info):
    """
    Table 1 재현: 월별 횡단면 통계의 시계열 평균

    Args:
        mler         : MLER DataFrame (행=날짜, 열=종목)
        periods_info : [(label, start, end), ...]

    Returns:
        DataFrame: Table 1 형식
    """
    rows = []

    for label, start, end in periods_info:
        # 해당 기간 슬라이싱
        subset = mler.loc[start:end]

        # 매월 횡단면 통계 계산
        monthly_stats = []
        for date, row in subset.iterrows():
            valid = row.dropna()
            if len(valid) == 0:
                continue
            monthly_stats.append({
                "Mean"  : valid.mean(),
                "S.D."  : valid.std(),
                "Min."  : valid.min(),
                "P1"    : valid.quantile(0.01),
                "P5"    : valid.quantile(0.05),
                "P25"   : valid.quantile(0.25),
                "Median": valid.quantile(0.50),
                "P75"   : valid.quantile(0.75),
                "P95"   : valid.quantile(0.95),
                "P99"   : valid.quantile(0.99),
                "Max."  : valid.max(),
                "n"     : len(valid),
            })

        if not monthly_stats:
            continue

        # 시계열 평균
        stats_df = pd.DataFrame(monthly_stats)
        avg      = stats_df.mean()

        rows.append({
            "Period": label,
            "Mean"  : round(avg["Mean"],   2),
            "S.D."  : round(avg["S.D."],   2),
            "Min."  : round(avg["Min."],   2),
            "P1"    : round(avg["P1"],     2),
            "P5"    : round(avg["P5"],     2),
            "P25"   : round(avg["P25"],    2),
            "Median": round(avg["Median"], 2),
            "P75"   : round(avg["P75"],    2),
            "P95"   : round(avg["P95"],    2),
            "P99"   : round(avg["P99"],    2),
            "Max."  : round(avg["Max."],   2),
            "n"     : round(avg["n"],      0),
        })

    table1 = pd.DataFrame(rows).set_index("Period")
    return table1


# 한국 샘플 기간 정의 (논문 Panel B 기준)
korean_periods = [
    ("Full sample:\n2003:01–2023:12",  "2003-01-31", "2023-12-31"),
    ("Sub-period:\n2003:01–2012:12",   "2003-01-31", "2012-12-31"),
    ("Sub-period:\n2013:01–2023:12",   "2013-01-31", "2023-12-31"),
]

table1 = compute_table1(mler, korean_periods)

print("Table 1: ML-Based Forecast Summary Statistics")
print("Panel B: Korean sample")
print("=" * 95)
print(table1.to_string())
print("=" * 95)
print("\n※ 각 셀 값은 월별 횡단면 통계의 시계열 평균")


Table 1: ML-Based Forecast Summary Statistics
Panel B: Korean sample
                               Mean  S.D.  Min.    P1    P5   P25  Median   P75   P95   P99  Max.       n
Period                                                                                                   
Full sample:\n2003:01–2023:12  0.01  0.04 -0.35 -0.12 -0.05 -0.01    0.01  0.03  0.06  0.07  0.10  1822.0
Sub-period:\n2003:01–2012:12   0.00  0.02 -0.22 -0.08 -0.04 -0.01    0.00  0.01  0.02  0.03  0.05  1596.0
Sub-period:\n2013:01–2023:12   0.02  0.05 -0.46 -0.15 -0.06 -0.00    0.02  0.05  0.09  0.11  0.16  2026.0

※ 각 셀 값은 월별 횡단면 통계의 시계열 평균


In [152]:
monthly_return = pd.read_csv(
        r"data\monthly_return.csv",
        index_col="Date",
        parse_dates=True
    )
market_cap = pd.read_csv(
        r"data\mkt_cap.csv",
        index_col="Date",
        parse_dates=True
    )
cd91 = pd.read_csv(
        r"data\cd91.csv",
        index_col="Date",
        parse_dates=True
    )
activate = pd.read_csv(
        r"data\activate.csv",
        index_col="Date",
        parse_dates=True
    )

monthly_return = monthly_return / 100
cd91           = cd91 / 100

risk_free = (1 + cd91) ** (1 / 12) - 1

# 거래대금 하위 10% 필터링 계산 데이터
trading_value_60 = pd.read_csv("./data/trading value 60.csv", index_col=0, parse_dates=True)  # 60영업일 평균 거래대금

# Amihud illiquidity 계산 데이터
daily_ret        = pd.read_csv("./data/daily ret.csv", index_col=0, parse_dates=True)         # 일간 수익률 월별 데이터
trading_value    = pd.read_csv("./data/trading value.csv", index_col=0, parse_dates=True)     # 월말 거래대금

C:\Users\bullh\AppData\Local\Temp\ipykernel_26504\2258957430.py:16: DtypeWarning:

Columns (57,88,97,116,127,132,142,149,157,193,220,221,224,237,239,258,262,271,285,322,348,353,367,370,374,383,396,398,403,415,437,443,476,488,498,504,506,513,524,534,560,564,565,575,606,613,615,624,649,675,682,696,701,712,713,731,747,757,771,779,795,798,806,807,812,815,879,896,903,910,930,968,974,991,1000,1008,1010,1015,1018,1021,1036,1041,1049,1051,1055,1088,1104,1115,1129,1153,1161,1162,1168,1231,1334,1338,1341,1345,1356,1379,1409,1410,1411,1430,1448,1472,1485,1515,1539,1560,1582,1588,1590,1594,1595,1610,1637,1677,1679,1750,1783,1803,1825,1859,1863,1873,1897,1904,1905,1915,1917,1925,1938,1960,1975,1986,1998,2000,2004,2031,2048,2087,2090,2108,2123,2129,2131,2164,2196,2223,2228,2229,2247,2275,2290,2304,2339,2358,2398,2513,2516,2524,2525,2528,2533,2545,2546,2547,2549,2553,2554,2555,2558,2560,2561,2562,2563,2565,2566,2567,2568,2569,2571,2573,2574,2575,2576,2578,2579,2580,2582,2583,2585,2586,2587,2588,2589,

In [153]:
from joblib import Parallel, delayed
from tqdm import tqdm

# 함수 정의
def run_strategy(wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, n_temp):
    
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    portfolio_name   = f"{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_Q({n_temp+1}/{q_cut})"
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )

    prev_portfolio = None
    
    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 거래대금 하위 x% 필터링
        series        = trading_value_60.loc[start_date]
        threshold     = series.quantile(trading_threshold_temp)
        filtered      = series[series > threshold]
        # factor_df(mler)에 없는 종목 제외
        common_stocks   = filtered.index.intersection(factor_df.columns)
        factor_filtered = factor_df.loc[start_date, common_stocks].dropna()

        # 종목 선정 (분위수)
        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
        basket    = factor_filtered[quantiles == n_temp]

        if basket.empty:
            continue

        # Amihud illiquidity 계산
        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        # 이전 비중
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        # 목표 비중
        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = market_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        # 거래비용 반영
        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        # 포트폴리오 가치
        current_portfolio_value = target_weights * NAV_new
        ret_seg                 = ret_wins.loc[end_date, basket.index]
        next_portfolio_value    = current_portfolio_value * (ret_seg + 1)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade

# 포트폴리오 선택 함수
def select_columns(df, fixed_options):
    selected = []
    new_names = {}
    
    for c in df.columns:
        parts = c.split('_')
        
        # 조건 체크 (고정한 옵션은 정확히 일치해야 함)
        if all(parts[i] == v or v == "*" for i, v in fixed_options.items()):
            selected.append(c)
            
            # 새 이름은 "고정하지 않은 옵션만"
            free_parts = [parts[i] for i, v in fixed_options.items() if v == "*"]
            new_name = "_".join(free_parts)
            new_names[c] = new_name
    
    # 선택된 subset 반환, 컬럼명은 자유 요소만
    subset = df[selected].rename(columns=new_names)
    return subset

In [154]:
# 팩터 설정
factor_df         = mler

start_point       = '2002-12-31'                     # 백테스트 기간 설정
end_point         = '2023-12-31'

q_cut             = 10                                # 포트폴리오를 나누는 분위 수 e.g. 10 → 10분위 
n                 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # 0 : 하위 ~ (q_cut - 1) : 상위

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']

cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)            e.g. [(0.008, 0.003), (0, 0)]
                                                     
wins_threshold    = [0.01, 0.00]                     # 수익률 윈저라이징                 e.g. [0.01, 0.05]
trading_threshold = [0.1, 0.0]                       # 60일 평균 거래대금 필터링         e.g. [0.1, 0.0]
Amihud_threshold  = [0.2, 0.0]                       # 유동성 기준                       e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

ret_m        = monthly_return

portfolio_df = pd.DataFrame()
trade_df     = pd.DataFrame()

month_ends   = factor_df.resample('ME').last().index
month_ends   = month_ends[(month_ends >= pd.to_datetime(start_point)) &
                        (month_ends <= pd.to_datetime(end_point))]

In [155]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(*params) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df = pd.DataFrame({r[0].name: r[0] for r in results})
trade_df     = pd.DataFrame({r[1].name: r[1] for r in results})

100%|██████████| 320/320 [05:59<00:00,  1.12s/it]


In [156]:
portfolio_df.tail()

,Equal_R0.01_H80L30_T0.1_A0.2_Q(1/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(2/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(3/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(4/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(5/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(6/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(7/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(8/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(9/10),Equal_R0.01_H80L30_T0.1_A0.2_Q(10/10),...,Cap_R0.0_H0L0_T0.0_A0.0_Q(1/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(2/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(3/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(4/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(5/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(6/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(7/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(8/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(9/10),Cap_R0.0_H0L0_T0.0_A0.0_Q(10/10)
2023-08-31,0.010100,0.009270,0.011901,0.023561,0.022927,0.022041,-0.004782,0.002049,0.008987,0.013350,...,-0.025727,-0.074573,-0.018918,0.000991,-0.009282,-0.013820,-0.023770,-0.037202,-0.027005,-0.032135
2023-09-30,-0.069872,-0.036471,-0.038339,-0.029589,-0.033137,-0.030291,-0.026262,-0.040112,-0.025870,-0.020326,...,-0.117884,-0.017018,-0.022848,-0.004654,-0.042996,-0.018058,-0.039963,-0.061535,-0.018414,0.020385
2023-10-31,-0.143146,-0.114843,-0.107373,-0.072540,-0.083182,-0.059609,-0.072972,-0.087806,-0.089104,-0.061116,...,-0.199205,-0.123484,-0.112708,-0.038786,-0.083459,-0.045367,-0.095321,-0.112146,-0.114740,-0.035025
2023-11-30,0.076431,0.072874,0.073416,0.060101,0.062031,0.072625,0.074635,0.058740,0.047864,0.057532,...,0.120522,0.161186,0.122692,0.068941,0.137259,0.092018,0.079377,0.060862,0.062803,0.087810
2023-12-31,-0.000253,0.028714,0.024589,0.025462,0.016867,0.037638,0.042591,0.037601,0.031166,0.013888,...,0.025962,0.020235,0.083157,0.046964,0.035025,0.041625,0.042457,0.019038,0.078782,0.060333


---
## **1. 분위수 검증**

##### **포트폴리오 선택**

In [157]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H08L03", 3: "T0.1", 4:"A0.2", 5: "*"} # *은 모든이라는 뜻
)

In [158]:
subset.tail()


""
2023-08-31
2023-09-30
2023-10-31
2023-11-30
2023-12-31


##### **Code**

In [159]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav = (subset + 1).cumprod()
log_cum = np.log1p(subset).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 첫 번째 그래프
for col in nav.columns:
    fig.add_trace(
        go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col,
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# 두 번째 그래프
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col,
                   legendgroup=col, showlegend=False),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Portfolio Performance Comparison",
    height=500, width=1000,
    template="plotly_white"
)

fig.update_layout(
    shapes=[
        # 왼쪽 그래프 영역 (x=0~0.45, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        
        # 오른쪽 그래프 영역 (x=0.55~1, y=0~1)
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

In [160]:
# ============================================================
# Table 2: Portfolio Analysis (논문 Table 2 재현)
# ============================================================
# 논문 방법론:
#   - 매월 말 MLER 기준 10분위 포트폴리오 구성
#   - 시가총액 가중 초과수익률 (포트폴리오 수익률 - 무위험수익률)
#   - 10–1 포트폴리오: Long MLER 10(상위) - Short MLER 1(하위)
#   - 보고 항목: 평균 초과수익률(r), t-통계량, 표준편차(SD), 연환산 샤프비율
# ============================================================

import numpy as np

# ── 1) 해당 조건의 10분위 포트폴리오 선택 ─────────────────────
subset_t2 = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.0", 2: "H0L0", 3: "T0.0", 4: "A0.0", 5: "*"}
).sort_index()

# ── 2) 초과수익률 계산 ────────────────────────────────────────
rf_series = risk_free.iloc[:, 0].reindex(subset_t2.index)
excess_t2 = subset_t2.sub(rf_series, axis=0)

# ── 3) 10–1 롱숏 포트폴리오 추가 ─────────────────────────────
#   Long MLER 10 (최상위) – Short MLER 1 (최하위)
excess_t2["MLER\n10–1"] = excess_t2["Q(10/10)"] - excess_t2["Q(1/10)"]

# ── 4) 컬럼명 정리 ────────────────────────────────────────────
rename_map = {f"Q({i}/10)": f"MLER\n{i}" for i in range(1, 11)}
excess_t2 = excess_t2.rename(columns=rename_map)
col_order = [f"MLER\n{i}" for i in range(1, 11)] + ["MLER\n10–1"]
excess_t2 = excess_t2[col_order]

# ── 5) 통계량 계산 ────────────────────────────────────────────
table2_rows = {}
for col in excess_t2.columns:
    s = excess_t2[col].dropna()
    n_obs = len(s)
    mean_r = s.mean()
    sd_r   = s.std(ddof=1)
    t_val  = mean_r / (sd_r / np.sqrt(n_obs))
    sharpe = (mean_r / sd_r) * np.sqrt(12)

    table2_rows[col] = {
        "r"     : f"{mean_r * 100:.2f}",
        ""      : f"({t_val:.2f})",
        "SD"    : f"{sd_r * 100:.2f}",
        "Sharpe": f"{sharpe:.2f}",
    }

table2 = pd.DataFrame(table2_rows)
table2.index.name = "Value"

# ── 6) 출력 ───────────────────────────────────────────────────
print("Table 2: Portfolio Analysis")
print("Panel B: Korean sample")
print("=" * 130)
print(table2.to_string())
print("=" * 130)
print("\n※ r: 월평균 초과수익률(%), ( ): t-통계량, SD: 월별 표준편차(%), Sharpe: 연환산 샤프비율")

Table 2: Portfolio Analysis
Panel B: Korean sample
       MLER\n1 MLER\n2 MLER\n3 MLER\n4 MLER\n5 MLER\n6 MLER\n7 MLER\n8 MLER\n9 MLER\n10 MLER\n10–1
Value                                                                                             
r         0.17    0.64    1.04    0.69    0.77    0.61    0.61    0.68    0.79     0.62       0.45
        (0.31)  (1.28)  (2.19)  (1.58)  (1.83)  (1.48)  (1.57)  (1.76)  (2.32)   (1.67)     (0.95)
SD        8.87    8.00    7.54    6.88    6.68    6.54    6.18    6.16    5.41     5.86       7.43
Sharpe    0.07    0.28    0.48    0.35    0.40    0.32    0.34    0.38    0.51     0.37       0.21

※ r: 월평균 초과수익률(%), ( ): t-통계량, SD: 월별 표준편차(%), Sharpe: 연환산 샤프비율


---
## **2. Double sort : Size Control**

##### **함수**

In [161]:
def run_strategy(c_temp, n_temp,
                 wins_threshold_temp, weight_method_temp, cost_temp,
                 trading_threshold_temp, Amihud_threshold_temp, 
                 initial_NAV=1):
    high_cost, low_cost = cost_temp
    NAV = initial_NAV
    portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({n_temp+1}/{q_cut})"
    
    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)
    
    # 수익률 윈저라이징
    ret_wins = ret_m.clip(
        lower = ret_m.quantile(wins_threshold_temp),
        upper = ret_m.quantile(1 - wins_threshold_temp),
        axis  = 1
    )
    
    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        cap_series    = market_cap.loc[start_date]
        cap_quantiles = pd.qcut(cap_series, q=cap_cut, labels=False)
        cap_filtered  = cap_series[cap_quantiles == c_temp]

        series    = trading_value_60.loc[start_date, cap_filtered.index]
        threshold = series.quantile(trading_threshold_temp)
        filtered  = series[series > threshold]
        # factor_df(mler)에 없는 종목 제외
        common_stocks   = filtered.index.intersection(factor_df.columns)
        factor_filtered = factor_df.loc[start_date, common_stocks].dropna()

        quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
        basket    = factor_filtered[quantiles == n_temp]
        if basket.empty:
            continue

        illiq        = (daily_ret.loc[start_date].abs() / trading_value.loc[start_date])
        threshold    = illiq.quantile(1 - Amihud_threshold_temp)
        illiquid_top = illiq[illiq >= threshold].index

        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket.index)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1/len(basket), index=basket.index)
        else:  # Cap
            cap_seg = market_cap.loc[start_date, basket.index]
            target_weights = cap_seg / cap_seg.sum()

        all_index     = target_weights.index.union(prev_weights.index)
        target_w      = target_weights.reindex(all_index, fill_value=0)
        prev_w        = prev_weights.reindex(all_index, fill_value=0)
        delta_w       = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiquid_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()
        NAV_new       = NAV - trade_cost

        current_portfolio_value = target_weights * NAV_new
        ret_seg = ret_wins.loc[end_date, basket.index]
        next_portfolio_value = current_portfolio_value * (ret_seg + 1)

        NAV_new       = next_portfolio_value.sum()
        portfolio_ret = NAV_new / NAV - 1

        prev_portfolio = next_portfolio_value
        NAV            = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    return portfolio_return, total_trade


##### **계산 실행**

In [162]:
cap_cut           = 5                                # 포트폴리오를 나누는 분위 수 e.g. 10 → 10분위 
c                 = [0, 1, 2, 3, 4]

q_cut             = 5                                # MLER 분위 수 (5분위 → 5×5 이중 정렬)
n                 = [0, 1, 2, 3, 4]

In [163]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[0]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
trading_threshold_temp = trading_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [164]:
# --- 병렬 실행 ---
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        c_temp, n_temp,
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp,
        initial_NAV
    )
    for c_temp in c
    for n_temp in n
)

# --- 결과 합치기 ---
cap_cut_profolio_df = pd.DataFrame()
for (portfolio_return, total_trade) in results:
    cap_cut_profolio_df[portfolio_return.name] = portfolio_return

##### **Output**

In [165]:
cap_cut_profolio_df.tail()

,C(1/5)_Q(1/5),C(1/5)_Q(2/5),C(1/5)_Q(3/5),C(1/5)_Q(4/5),C(1/5)_Q(5/5),C(2/5)_Q(1/5),C(2/5)_Q(2/5),C(2/5)_Q(3/5),C(2/5)_Q(4/5),C(2/5)_Q(5/5),...,C(4/5)_Q(1/5),C(4/5)_Q(2/5),C(4/5)_Q(3/5),C(4/5)_Q(4/5),C(4/5)_Q(5/5),C(5/5)_Q(1/5),C(5/5)_Q(2/5),C(5/5)_Q(3/5),C(5/5)_Q(4/5),C(5/5)_Q(5/5)
2023-08-31,0.037760,0.028761,0.034141,-0.007042,-0.002186,0.000842,0.021453,0.023333,0.006317,0.028350,...,0.022588,0.015874,0.022367,0.009027,0.031859,-0.016211,0.005077,-0.012119,-0.006664,-0.010069
2023-09-30,-0.060644,-0.046535,-0.035004,-0.029681,-0.001476,-0.050259,-0.033196,-0.051495,-0.011421,-0.022700,...,-0.047982,-0.024431,-0.034844,-0.044163,-0.029441,-0.065070,-0.009222,-0.002419,-0.027878,0.005385
2023-10-31,-0.098064,-0.076419,-0.054351,-0.055972,-0.022436,-0.114969,-0.092149,-0.062004,-0.079940,-0.072168,...,-0.155980,-0.087902,-0.076516,-0.070638,-0.070954,-0.150612,-0.076493,-0.075564,-0.106954,-0.088291
2023-11-30,0.084540,0.068093,0.059928,0.059191,0.022504,0.048602,0.063947,0.094417,0.079949,0.055503,...,0.078658,0.059512,0.067744,0.068197,0.064001,0.112967,0.092773,0.090082,0.062278,0.071114
2023-12-31,-0.001110,0.014079,0.042410,0.005045,-0.010755,0.001810,0.023423,0.033949,0.028749,0.004157,...,0.036692,0.038315,0.036928,0.043101,0.035978,0.015103,0.037579,0.046456,0.046026,0.047797


##### **Code**

In [166]:
def CAGR(series, periods_per_year=12):
    """월별 수익률 시리즈 기준 CAGR 계산"""
    series = series.dropna()
    if series.empty:
        return np.nan
    cumulative = (1 + series).cumprod()
    start, end = cumulative.index[0], cumulative.index[-1]
    n_years = (end - start).days / 365.25
    return (cumulative.iloc[-1] ** (1 / n_years)) - 1

In [167]:
# heatmap용 DF 초기화
# cap_cut_profolio_df의 컬럼명 패턴에서 자동으로 cap_cut, q_cut 추출
_sample_col = cap_cut_profolio_df.columns[0]  # e.g. "C(1/10)_Q(1/10)"
_c_part, _q_part = _sample_col.split('_')
_cap_cut = int(_c_part.split('/')[1].rstrip(')'))
_q_cut   = int(_q_part.split('/')[1].rstrip(')'))

cagr_matrix = pd.DataFrame(
    index  =[f"C({i+1}/{_cap_cut})" for i in range(_cap_cut)],
    columns=[f"Q({j+1}/{_q_cut})"   for j in range(_q_cut)]
)

for col in cap_cut_profolio_df.columns:
    c_part, q_part = col.split('_')
    cagr_matrix.loc[c_part, q_part] = CAGR(cap_cut_profolio_df[col])

cagr_matrix = cagr_matrix.astype(float)
print(f"히트맵 크기: {cagr_matrix.shape}, NaN: {cagr_matrix.isna().sum().sum()}")
print(cagr_matrix)


히트맵 크기: (5, 5), NaN: 0
          Q(1/5)    Q(2/5)    Q(3/5)    Q(4/5)    Q(5/5)
C(1/5)  0.158879  0.228633  0.275324  0.228379  0.063269
C(2/5) -0.033701  0.108963  0.082753  0.113454  0.109713
C(3/5) -0.112805 -0.020033  0.037081  0.063707  0.087993
C(4/5) -0.106451 -0.037409  0.021459  0.054252  0.119816
C(5/5)  0.011074  0.053054  0.069815  0.050891  0.076606


In [168]:
import plotly.express as px

fig = px.imshow(
    cagr_matrix.astype(float),
    text_auto=".1%",
    color_continuous_scale="Blues",
    aspect="auto",
    labels=dict(
        x="Factor Decile (Q)", 
        y="Market Cap Decile (C)", 
        color="CAGR"
    )
)

fig.update_layout(
    title="CAGR Heatmap by Cap vs Factor Decile",
    xaxis=dict(side="top"),
    width=700,
    height=700,
    margin=dict(l=60, r=60, t=80, b=60)
)


In [169]:
individual20=pd.read_csv("data\indvidual20.csv",index_col='Date',parse_dates=True)

##### **Double Sort
개인 매수강도 = individual20 / market_cap

In [170]:
# ============================================================
# Double-sort 함수: CR_k × MLER → CAGR 행렬
# (거래대금 필터링 / Amihud 유동성 / 거래비용 / 윈저라이징 포함)
# ============================================================
def double_sort_cagr(cr_data, mler_df, ret_df, month_ends,
                     n_bins_cr=5, n_bins_mler=5,
                     # ── 필터 / 비용 파라미터 ──
                     trading_value_60_df=trading_value_60,   # 60일 평균 거래대금 DF
                     trading_threshold=0.1,                  # 거래대금 하위 필터 분위 (0.1 → 하위 10% 제거)
                     daily_ret_df=daily_ret,                 # 일간 수익률 DF (Amihud용)
                     trading_value_df=trading_value,         # 월말 거래대금 DF (Amihud용)
                     Amihud_threshold=0.2,                   # 비유동 상위 비율 (0.2 → 상위 20%)
                     high_cost=0.008,                        # 비유동 종목 거래비용
                     low_cost=0.003,                         # 일반 종목 거래비용
                     wins_threshold=0.01,                    # 수익률 윈저라이징 분위수 (0.01 → 1%/99%)
                     weight_method='Equal',                  # 'Equal' 또는 'Cap'
                     market_cap_df=market_cap):              # 시가총액 DF (Cap 가중용)
    """
    이중 정렬 포트폴리오 CAGR 계산
    (거래대금 필터, Amihud 유동성 비용, 윈저라이징 포함)

    Parameters
    ----------
    cr_data           : DataFrame – 1차 정렬 팩터 (행=날짜, 열=종목)
    mler_df           : DataFrame – 2차 정렬 팩터: MLER (행=날짜, 열=종목)
    ret_df            : DataFrame – 월별 수익률 (행=날짜, 열=종목)
    month_ends        : DatetimeIndex – 리밸런싱 날짜
    n_bins_cr         : int  – 1차 정렬 분위 수
    n_bins_mler       : int  – 2차 정렬(MLER) 분위 수
    trading_value_60_df : DataFrame – 60영업일 평균 거래대금 (None이면 필터 미적용)
    trading_threshold : float – 거래대금 하위 필터 분위수
    daily_ret_df      : DataFrame – 일간 |수익률| (Amihud용, None이면 미적용)
    trading_value_df  : DataFrame – 월말 거래대금 (Amihud용)
    Amihud_threshold  : float – 비유동 상위 비율
    high_cost         : float – 비유동 종목 편도 거래비용
    low_cost          : float – 일반 종목 편도 거래비용
    wins_threshold    : float – 윈저라이징 분위수 (양쪽)
    weight_method     : str  – 'Equal' 또는 'Cap'
    market_cap_df     : DataFrame – 시가총액 (Cap 가중 시 필요)

    Returns
    -------
    cagr_matrix : DataFrame (n_bins_cr × n_bins_mler)
    """
    # 수익률 윈저라이징
    if wins_threshold > 0:
        ret_wins = ret_df.clip(
            lower=ret_df.quantile(wins_threshold),
            upper=ret_df.quantile(1 - wins_threshold),
            axis=1
        )
    else:
        ret_wins = ret_df

    # 셀별 NAV 추적용 딕셔너리
    cell_nav  = {(b1, b2): 1.0 for b1 in range(n_bins_cr) for b2 in range(n_bins_mler)}
    cell_prev = {(b1, b2): None for b1 in range(n_bins_cr) for b2 in range(n_bins_mler)}
    cell_count = {(b1, b2): 0 for b1 in range(n_bins_cr) for b2 in range(n_bins_mler)}

    for i in range(len(month_ends) - 1):
        sort_date = month_ends[i]
        ret_date  = month_ends[i + 1]

        # 해당 월 데이터 존재 확인
        if sort_date not in cr_data.index or sort_date not in mler_df.index:
            continue
        if ret_date not in ret_wins.index:
            continue

        cr_row   = cr_data.loc[sort_date].dropna()
        mler_row = mler_df.loc[sort_date].dropna()
        ret_row  = ret_wins.loc[ret_date].dropna()

        # 공통 종목
        common = cr_row.index.intersection(mler_row.index).intersection(ret_row.index)

        # ── 거래대금 하위 필터링 ─────────────────────────────
        if trading_value_60_df is not None and trading_threshold > 0:
            if sort_date in trading_value_60_df.index:
                tv60 = trading_value_60_df.loc[sort_date].dropna()
                tv60_common = tv60.reindex(common).dropna()
                thr = tv60_common.quantile(trading_threshold)
                common = tv60_common[tv60_common > thr].index

        if len(common) < n_bins_cr * n_bins_mler * 2:
            continue

        cr_c   = cr_row.reindex(common).dropna()
        mler_c = mler_row.reindex(common).dropna()
        ret_c  = ret_row.reindex(common).dropna()
        common = cr_c.index.intersection(mler_c.index).intersection(ret_c.index)
        cr_c, mler_c, ret_c = cr_c[common], mler_c[common], ret_c[common]

        # ── Amihud 비유동 종목 판별 ──────────────────────────
        illiquid_top = pd.Index([])
        if (daily_ret_df is not None and trading_value_df is not None
                and Amihud_threshold > 0):
            if sort_date in daily_ret_df.index and sort_date in trading_value_df.index:
                illiq = daily_ret_df.loc[sort_date].abs() / trading_value_df.loc[sort_date]
                illiq = illiq.dropna()
                thr_illiq = illiq.quantile(1 - Amihud_threshold)
                illiquid_top = illiq[illiq >= thr_illiq].index

        # ── 1차 정렬 ────────────────────────────────────────
        try:
            q_cr = pd.qcut(cr_c, q=n_bins_cr, labels=False, duplicates='drop')
        except ValueError:
            continue

        # ── 2차 정렬 (1차 그룹 내) ───────────────────────────
        for b1 in range(n_bins_cr):
            mask1 = q_cr == b1
            if mask1.sum() < n_bins_mler * 2:
                continue

            mler_sub = mler_c[mask1]
            ret_sub  = ret_c[mask1]

            try:
                q_mler = pd.qcut(mler_sub, q=n_bins_mler, labels=False, duplicates='drop')
            except ValueError:
                continue

            for b2 in range(n_bins_mler):
                mask2 = q_mler == b2
                if mask2.sum() == 0:
                    continue

                basket = ret_sub[mask2].index
                key    = (b1, b2)
                NAV    = cell_nav[key]

                # ── 목표 비중 계산 ───────────────────────────
                if weight_method == 'Cap' and market_cap_df is not None:
                    if sort_date in market_cap_df.index:
                        cap_seg = market_cap_df.loc[sort_date, basket].dropna()
                        if cap_seg.sum() > 0:
                            target_w = cap_seg / cap_seg.sum()
                        else:
                            target_w = pd.Series(1 / len(basket), index=basket)
                    else:
                        target_w = pd.Series(1 / len(basket), index=basket)
                else:
                    target_w = pd.Series(1 / len(basket), index=basket)

                # ── 이전 비중 ────────────────────────────────
                prev_port = cell_prev[key]
                if prev_port is not None and prev_port.sum() > 0:
                    prev_w = prev_port / prev_port.sum()
                else:
                    prev_w = pd.Series(0.0, index=basket)

                # ── 거래비용 계산 ────────────────────────────
                if high_cost > 0 or low_cost > 0:
                    all_idx  = target_w.index.union(prev_w.index)
                    tw       = target_w.reindex(all_idx, fill_value=0)
                    pw       = prev_w.reindex(all_idx, fill_value=0)
                    delta    = tw - pw
                    trade_am = abs(delta) * NAV
                    cost_r   = np.where(delta.index.isin(illiquid_top), high_cost, low_cost)
                    cost_tot = (trade_am * cost_r).sum()
                else:
                    cost_tot = 0.0

                NAV_after_cost = NAV - cost_tot

                # ── 수익률 반영 ──────────────────────────────
                port_value = target_w * NAV_after_cost
                next_value = port_value * (1 + ret_sub[mask2].reindex(basket, fill_value=0))
                NAV_new    = next_value.sum()

                cell_nav[key]   = NAV_new
                cell_prev[key]  = next_value
                cell_count[key] += 1

    # ── CAGR 계산 ────────────────────────────────────────────
    cagr_matrix = pd.DataFrame(
        np.nan,
        index  =[f"CR({i+1}/{n_bins_cr})"   for i in range(n_bins_cr)],
        columns=[f"MLER({j+1}/{n_bins_mler})" for j in range(n_bins_mler)]
    )

    for (b1, b2), final_nav in cell_nav.items():
        n_months = cell_count[(b1, b2)]
        if n_months > 0 and final_nav > 0:
            n_years = n_months / 12
            cagr_matrix.iloc[b1, b2] = final_nav ** (1 / n_years) - 1

    return cagr_matrix.astype(float)

print("double_sort_cagr 함수 정의 완료 (필터/비용 포함)")

double_sort_cagr 함수 정의 완료 (필터/비용 포함)


In [171]:
# ============================================================
# 개인 매수강도 × MLER  5×5 Double Sort → CAGR 히트맵
# ============================================================
import plotly.graph_objects as go

N_BINS = 5

# ── 1) 개인 매수강도 계산 ────────────────────────────────────
# 공통 인덱스/컬럼으로 정렬하여 나눗셈
common_idx  = individual20.index.intersection(market_cap.index)
common_cols = individual20.columns.intersection(market_cap.columns)
buy_intensity = individual20.loc[common_idx, common_cols] / market_cap.loc[common_idx, common_cols]

print(f"개인 매수강도 shape: {buy_intensity.shape}")
print(f"기간: {buy_intensity.index[0].date()} ~ {buy_intensity.index[-1].date()}")
print(f"유효값 비율: {buy_intensity.notna().values.mean():.2%}\n")

# ── 2) double_sort_cagr 재활용 ───────────────────────────────
#   1차 정렬 = 개인 매수강도 (cr_data 인자에 전달)
#   2차 정렬 = MLER
cagr_ind_mler = double_sort_cagr(
    cr_data    = buy_intensity,
    mler_df    = mler,
    ret_df     = monthly_return,
    month_ends = month_ends,
    n_bins_cr  = N_BINS,
    n_bins_mler= N_BINS,
    # 필터 / 비용 파라미터
    trading_value_60_df = trading_value_60,
    trading_threshold   = 0.1,
    daily_ret_df        = daily_ret,
    trading_value_df    = trading_value,
    Amihud_threshold    = 0.2,
    high_cost           = 0.008,
    low_cost            = 0.003,
    wins_threshold      = 0.01,
    weight_method       = 'Equal',
    market_cap_df       = market_cap
)

# 라벨 수정 (CR → BI)
cagr_ind_mler.index   = [f"BI({i+1}/{N_BINS})" for i in range(N_BINS)]
cagr_ind_mler.columns = [f"MLER({j+1}/{N_BINS})" for j in range(N_BINS)]

print("CAGR 행렬:")
print(cagr_ind_mler.to_string(float_format="{:.2%}".format))

# ── 3) 히트맵 시각화 ─────────────────────────────────────────
text_vals = [[f"{v:.1%}" if not np.isnan(v) else "" for v in row]
             for row in cagr_ind_mler.values]

fig = go.Figure(data=go.Heatmap(
    z=cagr_ind_mler.values,
    x=cagr_ind_mler.columns.tolist(),
    y=cagr_ind_mler.index.tolist(),
    colorscale="Blues",
    text=text_vals,
    texttemplate="%{text}",
    textfont=dict(size=12),
    colorbar=dict(title="CAGR", tickformat=".1%"),
))

fig.update_layout(
    title="Double Sort: Individual Buy Intensity × MLER → CAGR (5×5)",
    xaxis_title="MLER Quintile (1=Low, 5=High)",
    yaxis_title="Buy Intensity Quintile (1=Low, 5=High)",
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    width=700,
    height=600,
    margin=dict(l=80, r=60, t=80, b=60)
)

fig.show()

개인 매수강도 shape: (253, 3912)
기간: 2002-12-31 ~ 2023-12-31
유효값 비율: 47.69%

CAGR 행렬:
         MLER(1/5)  MLER(2/5)  MLER(3/5)  MLER(4/5)  MLER(5/5)
BI(1/5)     -3.40%      5.97%      8.36%     12.41%      5.41%
BI(2/5)     -3.16%      6.07%      9.56%      6.72%      7.02%
BI(3/5)      0.89%      6.91%      7.38%     11.61%     13.06%
BI(4/5)     -5.31%     -1.33%      3.49%      1.52%      5.50%
BI(5/5)    -12.65%      0.55%      0.33%      5.34%      4.44%


In [172]:
# ============================================================
# 기관 매수강도 × MLER  5×5 Double Sort → CAGR 히트맵
# ============================================================
import plotly.graph_objects as go

N_BINS_INST = 5

# ── 0) 데이터 로드 ───────────────────────────────────────────
inst20 = pd.read_csv(r"data\inst20.csv", index_col="Date", parse_dates=True)

# ── 1) 기관 매수강도 계산 ────────────────────────────────────
common_idx_inst  = inst20.index.intersection(market_cap.index)
common_cols_inst = inst20.columns.intersection(market_cap.columns)
inst_buy_intensity = inst20.loc[common_idx_inst, common_cols_inst] / market_cap.loc[common_idx_inst, common_cols_inst]

print(f"기관 매수강도 shape: {inst_buy_intensity.shape}")
print(f"기간: {inst_buy_intensity.index[0].date()} ~ {inst_buy_intensity.index[-1].date()}")
print(f"유효값 비율: {inst_buy_intensity.notna().values.mean():.2%}\n")

# ── 2) double_sort_cagr 호출 ─────────────────────────────────
cagr_inst_mler = double_sort_cagr(
    cr_data    = inst_buy_intensity,
    mler_df    = mler,
    ret_df     = monthly_return,
    month_ends = month_ends,
    n_bins_cr  = N_BINS_INST,
    n_bins_mler= N_BINS_INST,
    trading_value_60_df = trading_value_60,
    trading_threshold   = 0.1,
    daily_ret_df        = daily_ret,
    trading_value_df    = trading_value,
    Amihud_threshold    = 0.2,
    high_cost           = 0.008,
    low_cost            = 0.003,
    wins_threshold      = 0.01,
    weight_method       = 'Equal',
    market_cap_df       = market_cap
)

# 라벨 수정 (CR → Inst BI)
cagr_inst_mler.index   = [f"Inst BI({i+1}/{N_BINS_INST})" for i in range(N_BINS_INST)]
cagr_inst_mler.columns = [f"MLER({j+1}/{N_BINS_INST})" for j in range(N_BINS_INST)]

print("CAGR 행렬:")
print(cagr_inst_mler.to_string(float_format="{:.2%}".format))

# ── 3) 히트맵 시각화 ─────────────────────────────────────────
text_vals_inst = [[f"{v:.1%}" if not np.isnan(v) else "" for v in row]
                  for row in cagr_inst_mler.values]

fig = go.Figure(data=go.Heatmap(
    z=cagr_inst_mler.values,
    x=cagr_inst_mler.columns.tolist(),
    y=cagr_inst_mler.index.tolist(),
    colorscale="Greens",
    text=text_vals_inst,
    texttemplate="%{text}",
    textfont=dict(size=12),
    colorbar=dict(title="CAGR", tickformat=".1%"),
))

fig.update_layout(
    title="Double Sort: Institutional Buy Intensity × MLER → CAGR (5×5)",
    xaxis_title="MLER Quintile (1=Low, 5=High)",
    yaxis_title="Inst. Buy Intensity Quintile (1=Low, 5=High)",
    yaxis=dict(autorange="reversed"),
    template="plotly_white",
    width=700,
    height=600,
    margin=dict(l=80, r=60, t=80, b=60)
)

fig.show()

기관 매수강도 shape: (253, 3912)
기간: 2002-12-31 ~ 2023-12-31
유효값 비율: 47.69%

CAGR 행렬:
              MLER(1/5)  MLER(2/5)  MLER(3/5)  MLER(4/5)  MLER(5/5)
Inst BI(1/5)     -5.26%      4.47%      6.21%      9.16%      7.32%
Inst BI(2/5)     -8.88%      2.81%      6.68%      6.84%      8.68%
Inst BI(3/5)      1.16%     11.63%     12.73%     15.34%     12.57%
Inst BI(4/5)     -3.58%      5.00%      8.25%     10.60%      8.34%
Inst BI(5/5)     -6.22%      0.36%      1.37%      4.42%      3.82%


In [173]:
# ============================================================
# CR1 ~ CR12 계산 (벡터 연산)
# ============================================================
# CR_k(t) = 과거 k개월 누적수익률 (t-k ~ t-1, 현재 월 미포함)

log_ret = np.log1p(monthly_return)

CR = {}
for k in range(1, 13):
    cr_k = np.expm1(log_ret.rolling(k, min_periods=k).sum().shift(1))
    CR[k] = cr_k
    n_valid = cr_k.notna().sum().sum()
    print(f"  CR{k:>2} 완료  (유효 셀: {n_valid:>10,})")

print("\nCR1 ~ CR12 계산 완료")

  CR 1 완료  (유효 셀:    604,546)
  CR 2 완료  (유효 셀:    600,924)
  CR 3 완료  (유효 셀:    597,312)
  CR 4 완료  (유효 셀:    593,715)
  CR 5 완료  (유효 셀:    590,130)
  CR 6 완료  (유효 셀:    586,559)
  CR 7 완료  (유효 셀:    583,024)
  CR 8 완료  (유효 셀:    579,499)
  CR 9 완료  (유효 셀:    575,984)
  CR10 완료  (유효 셀:    572,482)
  CR11 완료  (유효 셀:    568,991)
  CR12 완료  (유효 셀:    565,502)

CR1 ~ CR12 계산 완료


In [174]:
# ============================================================
# CR1~CR12 × MLER 이중 정렬 CAGR 히트맵 (3×4 그리드)
# ============================================================
from plotly.subplots import make_subplots
import plotly.graph_objects as go

N_BINS_CR   = 5   # CR 분위 수 (조절 가능)
N_BINS_MLER = 5   # MLER 분위 수 (조절 가능)

# ── 1) 12개 CAGR 행렬 계산 ──────────────────────────────────
cagr_matrices = {}
for k in range(1, 13):
    cagr_matrices[k] = double_sort_cagr(
        CR[k], mler, monthly_return, month_ends,
        n_bins_cr=N_BINS_CR, n_bins_mler=N_BINS_MLER,
        # 필터 / 비용 파라미터
        trading_value_60_df = trading_value_60,
        trading_threshold   = 0.1,
        daily_ret_df        = daily_ret,
        trading_value_df    = trading_value,
        Amihud_threshold    = 0.2,
        high_cost           = 0.008,
        low_cost            = 0.003,
        wins_threshold      = 0.01,
        weight_method       = 'Equal',
        market_cap_df       = market_cap
    )
    print(f"  CR{k:>2} × MLER 이중 정렬 완료")

# ── 2) 전체 공통 색상 범위 ───────────────────────────────────
all_vals  = np.concatenate([m.values.flatten() for m in cagr_matrices.values()])
all_vals  = all_vals[~np.isnan(all_vals)]
global_min, global_max = all_vals.min(), all_vals.max()

# ── 3) 3×4 서브플롯 히트맵 ──────────────────────────────────
fig = make_subplots(
    rows=3, cols=4,
    subplot_titles=[f"CR{k} × MLER" for k in range(1, 13)],
    horizontal_spacing=0.06,
    vertical_spacing=0.10
)

for k in range(1, 13):
    matrix = cagr_matrices[k]
    row = (k - 1) // 4 + 1
    col = (k - 1) % 4 + 1

    # 텍스트 생성 (퍼센트 표시)
    text_vals = [[f"{v:.1%}" if not np.isnan(v) else "" for v in r]
                 for r in matrix.values]

    fig.add_trace(
        go.Heatmap(
            z=matrix.values,
            x=matrix.columns.tolist(),
            y=matrix.index.tolist(),
            colorscale="Blues",
            zmin=global_min,
            zmax=global_max,
            text=text_vals,
            texttemplate="%{text}",
            textfont=dict(size=10),
            showscale=(k == 12),
            colorbar=dict(
                title="CAGR",
                tickformat=".1%",
                len=0.3, y=0.15
            ) if k == 12 else None,
        ),
        row=row, col=col
    )

fig.update_layout(
    title=dict(
        text="Double Sort: CR<sub>k</sub> × MLER → CAGR Heatmaps (k = 1, …, 12)",
        font=dict(size=16)
    ),
    height=900,
    width=1200,
    template="plotly_white",
    margin=dict(l=60, r=60, t=80, b=40)
)

# y축을 아래→위로 (CR 작은 값이 하단)
fig.update_yaxes(autorange="reversed")

fig.show()
print(f"\n분위 설정: CR {N_BINS_CR}분위 × MLER {N_BINS_MLER}분위")


  CR 1 × MLER 이중 정렬 완료
  CR 2 × MLER 이중 정렬 완료
  CR 3 × MLER 이중 정렬 완료
  CR 4 × MLER 이중 정렬 완료
  CR 5 × MLER 이중 정렬 완료
  CR 6 × MLER 이중 정렬 완료
  CR 7 × MLER 이중 정렬 완료
  CR 8 × MLER 이중 정렬 완료
  CR 9 × MLER 이중 정렬 완료
  CR10 × MLER 이중 정렬 완료
  CR11 × MLER 이중 정렬 완료
  CR12 × MLER 이중 정렬 완료



분위 설정: CR 5분위 × MLER 5분위


---
## **Double Sort: CR × MLER**
CR1~CR12 각각을 첫 번째 정렬 기준으로, MLER을 두 번째 정렬 기준으로 사용한 5×5 이중 정렬 CAGR 히트맵---
## **3. Parameter Sensitivity**
`0-2. 포트폴리오 계산`에서 계산한 포트폴리오 `portfolio_df` 데이터 사용

---
#### **1) Weight sensitivity**

##### **포트폴리오 선택**

In [175]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_equal = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

# 시총가중 포트폴리오
subset_cap = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

##### **Code**

In [176]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 로그누적수익률 계산
log_cum_equal = np.log1p(subset_equal).cumsum()
log_cum_cap   = np.log1p(subset_cap).cumsum()

# 서브플롯 (가로 2개)
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=["Equal-weighted Portfolios", 
                    "Cap-weighted Portfolios"]
)

# Equal 그래프
for col in log_cum_equal.columns:
    fig.add_trace(
        go.Scatter(x=log_cum_equal.index, y=log_cum_equal[col], 
                   mode='lines', name=f"Equal_{col}",
                   legendgroup=col, showlegend=True),
        row=1, col=1
    )

# Cap 그래프
for col in log_cum_cap.columns:
    fig.add_trace(
        go.Scatter(x=log_cum_cap.index, y=log_cum_cap[col], 
                   mode='lines', name=f"Cap_{col}",
                   legendgroup=col, showlegend=True),
        row=1, col=2
    )

# 레이아웃 기본 설정
fig.update_layout(
    title="Equal vs Cap Weighted Portfolios (Log Cumulative Returns)",
    height=500, width=1000,
    template="plotly_white"
)

# 각 subplot 테두리 표시
fig.update_layout(
    shapes=[
        dict(type="rect", xref="paper", yref="paper",
             x0=0.0, y0=0.0, x1=0.45, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
        dict(type="rect", xref="paper", yref="paper",
             x0=0.55, y0=0.0, x1=1.0, y1=1.0,
             line=dict(color="gray", width=1), fillcolor="rgba(0,0,0,0)"),
    ]
)

---
#### **2) Cost sensitivity**

##### **포트폴리오 선택**

In [177]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_cost = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.0", 2: "*", 3: "T0.0", 4:"A0.0", 5: "Q(10/10)"}
)

##### **Code**

In [178]:
import numpy as np
import plotly.graph_objects as go

# 로그누적수익률 계산
log_cum_cost = np.log1p(subset_cost).cumsum()

# 단일 plot 생성
fig = go.Figure()

for col in log_cum_cost.columns:
    fig.add_trace(
        go.Scatter(
            x=log_cum_cost.index, 
            y=log_cum_cost[col], 
            mode='lines', 
            name=col
        )
    )

# 레이아웃 설정
fig.update_layout(
    title="Equal-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    xaxis_title="Date",
    yaxis_title="Log Cumulative Return",
    template="plotly_white",
    height=600,
    width=900
)

In [179]:
# 요청한 포트폴리오 선택
subset_kospi_cmp = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "*", 3: "T0.1", 4: "A0.2", 5: "Q(10/10)"}
)

# 코스피 수익률 시리즈 준비
if "kospi" in globals():
    kospi_obj = kospi.copy()
    if isinstance(kospi_obj, pd.DataFrame):
        kospi_obj = kospi_obj.iloc[:, 0]
    # 값 크기로 지수레벨/수익률 구분
    kospi_ret = kospi_obj.pct_change() if kospi_obj.dropna().median() > 2 else kospi_obj
elif any(("코스피" in c) or ("KOSPI" in c.upper()) for c in monthly_return.columns):
    kospi_col = [c for c in monthly_return.columns if ("코스피" in c) or ("KOSPI" in c.upper())][0]
    kospi_ret = monthly_return[kospi_col]
else:
    # 파일 fallback (지수 레벨 데이터 가정)
    kospi_df = pd.read_csv("./data/kospi.csv", index_col=0, parse_dates=True)
    kospi_ser = kospi_df.iloc[:, 0]
    kospi_ret = kospi_ser.pct_change() if kospi_ser.dropna().median() > 2 else kospi_ser

# 인덱스 정렬 및 로그누적수익률 계산
plot_df = subset_kospi_cmp.copy()
plot_df["KOSPI"] = kospi_ret
plot_df = plot_df.sort_index().dropna(how="all")

log_cum = np.log1p(plot_df).cumsum()

# 그래프
fig = go.Figure()
for col in log_cum.columns:
    fig.add_trace(
        go.Scatter(
            x=log_cum.index,
            y=log_cum[col],
            mode="lines",
            name=col,
            line=dict(width=3 if col == "KOSPI" else 2)
        )
    )

fig.update_layout(
    title='KOSPI vs Portfolio (Equal, R0.01, Cost=*, T0.1, A0.2, Q(1/5)) - Log Cumulative Return',
    xaxis_title="Date",
    yaxis_title="Log Cumulative Return",
    template="plotly_white",
    height=600,
    width=1000
)

---
## **Excel Export: 주요 분석 결과 저장**

In [181]:
import os
import pandas as pd
import numpy as np

save_dir = "final"
os.makedirs(save_dir, exist_ok=True)

# ============================================================
# 1) Table 1: MLER 요약 통계
# ============================================================
table1.to_excel(os.path.join(save_dir, "table1_mler_summary.xlsx"))
print("✅ Table 1 저장 → final/table1_mler_summary.xlsx")

# ============================================================
# 2) Table 2: 10분위 포트폴리오 수익률
# ============================================================
with pd.ExcelWriter(os.path.join(save_dir, "table2_portfolio.xlsx")) as writer:
    table2.to_excel(writer, sheet_name="Table2_Formatted")
    excess_t2.to_excel(writer, sheet_name="Excess_Return_Raw")
print("✅ Table 2 저장 → final/table2_portfolio.xlsx")

# ============================================================
# 3) NAV / 로그누적수익률 차트 데이터
# ============================================================
with pd.ExcelWriter(os.path.join(save_dir, "nav_logcum.xlsx")) as writer:
    nav.to_excel(writer, sheet_name="NAV")
    _log_cum_chart = np.log1p(subset).cumsum()
    _log_cum_chart.to_excel(writer, sheet_name="Log_Cumulative_Return")
print("✅ NAV / 로그누적수익률 저장 → final/nav_logcum.xlsx")

# ============================================================
# 4) Size × MLER 5×5 이중정렬 CAGR 히트맵
# ============================================================
with pd.ExcelWriter(os.path.join(save_dir, "size_mler_double_sort.xlsx")) as writer:
    cagr_matrix.to_excel(writer, sheet_name="CAGR_Heatmap")
    cap_cut_profolio_df.to_excel(writer, sheet_name="Portfolio_Returns_Raw")
print("✅ Size × MLER 저장 → final/size_mler_double_sort.xlsx")

# ============================================================
# 5) CR1~CR12 × MLER 이중정렬 CAGR 히트맵
# ============================================================
with pd.ExcelWriter(os.path.join(save_dir, "cr_mler_double_sort.xlsx")) as writer:
    for k in range(1, 13):
        cagr_matrices[k].to_excel(writer, sheet_name=f"CR{k}_x_MLER")
print("✅ CR1~CR12 × MLER 저장 → final/cr_mler_double_sort.xlsx")

# ============================================================
# 6) 개인 매수강도 × MLER 히트맵
# ============================================================
cagr_ind_mler.to_excel(os.path.join(save_dir, "individual_bi_mler.xlsx"))
print("✅ 개인 매수강도 × MLER 저장 → final/individual_bi_mler.xlsx")

# ============================================================
# 7) 기관 매수강도 × MLER 히트맵
# ============================================================
cagr_inst_mler.to_excel(os.path.join(save_dir, "institutional_bi_mler.xlsx"))
print("✅ 기관 매수강도 × MLER 저장 → final/institutional_bi_mler.xlsx")

# ============================================================
# 8) Parameter Sensitivity
# ============================================================
with pd.ExcelWriter(os.path.join(save_dir, "parameter_sensitivity.xlsx")) as writer:
    # 8-1) Weight sensitivity: Equal vs Cap
    subset_equal.to_excel(writer, sheet_name="Equal_Weight_Returns")
    subset_cap.to_excel(writer, sheet_name="Cap_Weight_Returns")
    log_cum_equal.to_excel(writer, sheet_name="Equal_LogCum")
    log_cum_cap.to_excel(writer, sheet_name="Cap_LogCum")
    # 8-2) Cost sensitivity
    subset_cost.to_excel(writer, sheet_name="Cost_Sensitivity_Returns")
    log_cum_cost.to_excel(writer, sheet_name="Cost_LogCum")
    # 8-3) KOSPI comparison
    plot_df.to_excel(writer, sheet_name="KOSPI_Comparison_Returns")
    log_cum.to_excel(writer, sheet_name="KOSPI_Comparison_LogCum")
print("✅ Parameter Sensitivity 저장 → final/parameter_sensitivity.xlsx")

print("\n" + "=" * 60)
print("모든 Excel 파일 저장 완료! (final/ 폴더)")
print("=" * 60)

✅ Table 1 저장 → final/table1_mler_summary.xlsx
✅ Table 2 저장 → final/table2_portfolio.xlsx
✅ NAV / 로그누적수익률 저장 → final/nav_logcum.xlsx
✅ Size × MLER 저장 → final/size_mler_double_sort.xlsx
✅ CR1~CR12 × MLER 저장 → final/cr_mler_double_sort.xlsx
✅ 개인 매수강도 × MLER 저장 → final/individual_bi_mler.xlsx
✅ 기관 매수강도 × MLER 저장 → final/institutional_bi_mler.xlsx
✅ Parameter Sensitivity 저장 → final/parameter_sensitivity.xlsx

모든 Excel 파일 저장 완료! (final/ 폴더)
